This is Real Model

# Job Recommendation System

A two-phase TF-IDF–based system:
- **Phase 1 – Job-to-Job similarity**: find jobs similar to a given posting.
- **Phase 2 – Resume-to-Job matching**: rank candidates against every job.

## 1. Imports & NLTK setup

In [ ]:
import re
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("All imports successful.")

## 2. Text preprocessing

In [ ]:
_stop_words = set(stopwords.words('english'))
_lemmatizer = WordNetLemmatizer()


def preprocess(text: str) -> str:
    """Clean and normalise a text string in a single pass.

    Steps:
    1. Lower-case
    2. Strip non-alpha characters
    3. Collapse whitespace
    4. Remove English stop-words
    5. Lemmatise each token

    Returns an empty string for missing / non-string input.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # keep only letters + spaces
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = [
        _lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in _stop_words and len(w) > 1   # drop single-char noise
    ]
    return ' '.join(tokens)

## 3. Load & prepare job data

In [ ]:
jobs_df = pd.read_csv("clean_job_dataset.csv")

# Build processed_text from Job Description if not already present
if 'processed_text' not in jobs_df.columns:
    jobs_df['processed_text'] = jobs_df['Job Description'].apply(preprocess)

# Drop rows with empty processed text and reset index
jobs_df = (
    jobs_df[jobs_df['processed_text'].str.strip().astype(bool)]
    .reset_index(drop=True)
)

print(f"Loaded {len(jobs_df):,} job postings.")
jobs_df.head()

## 4. Phase 1 – Baseline: Job-to-Job similarity

Vectorise all job descriptions with TF-IDF (unigrams + bigrams, 5 000 features)  
and compute pairwise similarity with `linear_kernel`, which is equivalent to  
cosine similarity on L2-normalised TF-IDF vectors but ~2× faster on sparse matrices.

In [ ]:
vectorizer_p1 = TfidfVectorizer(
    max_features=3000,     # was 100 — far too low to capture vocabulary
    ngram_range=(1, 2),     # unigrams + bigrams for richer matching
    sublinear_tf=True,      # apply log(1 + tf) to dampen high-frequency terms
    stop_words='english',
)

job_tfidf = vectorizer_p1.fit_transform(jobs_df['processed_text'])

# linear_kernel on L2-normalised sparse matrices == cosine similarity, but faster
job_sim_matrix = linear_kernel(job_tfidf, job_tfidf)

print(f"TF-IDF matrix shape: {job_tfidf.shape}")
print(f"Similarity matrix shape: {job_sim_matrix.shape}")

In [ ]:
def recommend_jobs(job_index: int, top_n: int = 5) -> pd.DataFrame:
    """Return the top-N most similar jobs to the job at `job_index`.

    Parameters
    ----------
    job_index : int
        Row index in jobs_df.
    top_n : int
        Number of recommendations to return (default 5).

    Returns
    -------
    pd.DataFrame with columns [Job Title, Company, similarity_score].
    """
    scores = job_sim_matrix[job_index]
    # argsort ascending → reverse → skip self (index 0)
    top_idx = scores.argsort()[::-1][1: top_n + 1]

    result = jobs_df.iloc[top_idx][['Job Title', 'Company']].copy()
    result['similarity_score'] = scores[top_idx].round(4)
    result.index = range(1, len(result) + 1)   # 1-based ranking
    return result

In [ ]:
# --- Test Phase 1 ---
seed_job = 0
print(f"Seed job: '{jobs_df.iloc[seed_job]['Job Title']}' @ {jobs_df.iloc[seed_job]['Company']}\n")
recommend_jobs(seed_job, top_n=5)

## 5. Phase 2 – Advanced: Resume-to-Job matching

Fit a shared TF-IDF vocabulary on **all** text (resumes + jobs), then  
compute a cross-similarity matrix **(resumes × jobs)**.  
Results are built with vectorised NumPy ops instead of a Python loop.

In [ ]:
resume_data = [
    {"name": "Candidate 1", "resume_text": "Python developer with machine learning and NLP experience"},
    {"name": "Candidate 2", "resume_text": "Java backend developer with Spring Boot and SQL"},
    {"name": "Candidate 3", "resume_text": "Data analyst with Excel, SQL and Power BI"},
    {"name": "Candidate 4", "resume_text": "AI engineer with deep learning and PyTorch"},
    {"name": "Candidate 5", "resume_text": "Frontend developer with React and JavaScript"},
]

resume_df = pd.DataFrame(resume_data)
resume_df['processed_text'] = resume_df['resume_text'].apply(preprocess)
resume_df[['name', 'processed_text']]

In [ ]:
vectorizer_p2 = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
)

# Fit on the union of resumes + jobs so both share the same feature space
combined_text = resume_df['processed_text'].tolist() + jobs_df['processed_text'].tolist()
tfidf_matrix = vectorizer_p2.fit_transform(combined_text)

n_resumes = len(resume_df)
resume_vectors = tfidf_matrix[:n_resumes]          # shape: (R, vocab)
job_vectors    = tfidf_matrix[n_resumes:]           # shape: (J, vocab)

# Cross-similarity: (R, J)  — each row = one resume, each col = one job
resume_job_sim = cosine_similarity(resume_vectors, job_vectors)

print(f"Resume-Job similarity matrix: {resume_job_sim.shape}  (resumes × jobs)")

In [ ]:
# Vectorised ranking: for every job find the top-3 candidates
TOP_K = 3

# Argsort descending along the resume axis (axis=0)
top_candidate_idx = np.argsort(-resume_job_sim, axis=0)[:TOP_K]   # (TOP_K, J)

records = []
for j in range(len(jobs_df)):
    for rank, c_idx in enumerate(top_candidate_idx[:, j], start=1):
        records.append({
            'job_title':  jobs_df.iloc[j]['Job Title'],
            'company':    jobs_df.iloc[j]['Company'],
            'rank':       rank,
            'candidate':  resume_df.iloc[c_idx]['name'],
            'score':      round(resume_job_sim[c_idx, j], 4),
        })

results_df = pd.DataFrame(records)
print(f"Total match records: {len(results_df):,}")
results_df.head(15)

In [ ]:
# Best (rank-1) candidate per job, sorted by match score descending
best_candidates = (
    results_df[results_df['rank'] == 1]
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

best_candidates.head(10)

## 6. Top jobs per candidate

In [ ]:
def top_jobs_for_candidate(candidate_name: str, top_n: int = 5) -> pd.DataFrame:
    """Return the best-matching jobs for a given candidate."""
    c_idx = resume_df.index[resume_df['name'] == candidate_name].item()
    scores = resume_job_sim[c_idx]                  # shape: (J,)
    top_idx = np.argsort(-scores)[:top_n]

    out = jobs_df.iloc[top_idx][['Job Title', 'Company']].copy()
    out['match_score'] = scores[top_idx].round(4)
    out.index = range(1, len(out) + 1)
    return out


for candidate in resume_df['name']:
    print(f"\n=== Top 3 jobs for {candidate} ===")
    print(top_jobs_for_candidate(candidate, top_n=3).to_string())